In [2]:
import pandas as pd
from scipy.stats import pearsonr

# Load the actual master dataset
df = pd.read_csv("monthly_ml_master_2025.csv")

# Check the dataset
print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

Dataset shape: (12, 25)

Columns:
['Year', 'Month_Num', 'Month', 'Quarter', 'ATM_Onsite', 'ATM_Offsite', 'POS', 'Micro_ATM', 'Bharat_QR', 'UPI_QR', 'Credit_Cards', 'Debit_Cards', 'CC_POS_Volume', 'CC_Online_Volume', 'DC_POS_Volume', 'DC_Online_Volume', 'DC_ATM_Withdrawal_Volume', 'PSI_Volume_Current_Month', 'PSI_Volume_Previous_Month', 'PSI_Volume_Same_Month_Last_Year', 'PSI_Value_Current_Month', 'PSI_Value_Previous_Month', 'PSI_Value_Same_Month_Last_Year', 'NPCI_Volume_Mn', 'NPCI_Value_Bn']


In [3]:
# Check the variables required for Hypothesis Test 1

variables = [
    "DC_ATM_Withdrawal_Volume",
    "NPCI_Volume_Mn"
]

print("HYPOTHESIS TEST 1 — DATA CHECK")
print("=" * 50)

# Check missing values
print("\nMissing values:")
print(df[variables].isnull().sum())

# Basic descriptive statistics
print("\nDescriptive Statistics:")
print(df[variables].describe())

# Display the actual observations
print("\nMonthly observations:")
print(df[["Month", "Quarter"] + variables].to_string(index=False))

HYPOTHESIS TEST 1 — DATA CHECK

Missing values:
DC_ATM_Withdrawal_Volume    0
NPCI_Volume_Mn              0
dtype: int64

Descriptive Statistics:
       DC_ATM_Withdrawal_Volume  NPCI_Volume_Mn
count              1.200000e+01       12.000000
mean               4.559295e+08   143361.739022
std                1.937341e+07     7725.176566
min                4.358581e+08   132106.807145
25%                4.428070e+08   139264.970094
50%                4.525914e+08   144944.161188
75%                4.610109e+08   149040.930115
max                4.957215e+08   151451.826569

Monthly observations:
    Month Quarter  DC_ATM_Withdrawal_Volume  NPCI_Volume_Mn
  January      Q1                 488353232   132106.807145
 February      Q1                 447376890   132106.807145
    March      Q1                 495721528   132106.807145
    April      Q2                 457805976   141651.024411
      May      Q2                 457987566   141651.024411
     June      Q2                 43585

In [4]:
# ============================================================
# HYPOTHESIS TEST 1 — STEP 3
# CREATE QUARTERLY DATA
# ============================================================

# Aggregate monthly DC ATM Withdrawal Volume to quarterly total
quarterly_ht1 = (
    df.groupby("Quarter")
      .agg(
          DC_ATM_Withdrawal_Volume=(
              "DC_ATM_Withdrawal_Volume", "sum"
          ),
          NPCI_Volume_Mn=(
              "NPCI_Volume_Mn", "first"
          )
      )
      .reset_index()
)

# Arrange quarters in correct order
quarter_order = ["Q1", "Q2", "Q3", "Q4"]

quarterly_ht1["Quarter"] = pd.Categorical(
    quarterly_ht1["Quarter"],
    categories=quarter_order,
    ordered=True
)

quarterly_ht1 = quarterly_ht1.sort_values("Quarter")

print("QUARTERLY DATA FOR HYPOTHESIS TEST 1")
print("=" * 60)

print(quarterly_ht1.to_string(index=False))

print("\nNumber of quarterly observations:",
      len(quarterly_ht1))

print("\nUnique NPCI Volume values:",
      quarterly_ht1["NPCI_Volume_Mn"].nunique())

QUARTERLY DATA FOR HYPOTHESIS TEST 1
Quarter  DC_ATM_Withdrawal_Volume  NPCI_Volume_Mn
     Q1                1431451650   132106.807145
     Q2                1351651681   141651.024411
     Q3                1341992348   148237.297964
     Q4                1346057982   151451.826569

Number of quarterly observations: 4

Unique NPCI Volume values: 4


In [5]:
# ============================================================
# HYPOTHESIS TEST 1 — STEP 4
# PEARSON CORRELATION TEST
# ============================================================

from scipy.stats import pearsonr

# Variables
x = quarterly_ht1["DC_ATM_Withdrawal_Volume"]
y = quarterly_ht1["NPCI_Volume_Mn"]

# Pearson correlation
r, p_value = pearsonr(x, y)

alpha = 0.05

print("HYPOTHESIS TEST 1 — PEARSON CORRELATION")
print("=" * 60)

print("\nResearch Question:")
print("Is there a statistically significant linear relationship")
print("between DC ATM Withdrawal Volume and NPCI Volume?")

print("\nH0:")
print("There is no statistically significant linear relationship.")

print("\nH1:")
print("There is a statistically significant linear relationship.")

print("\nResults:")
print(f"Pearson correlation (r): {r:.4f}")
print(f"P-value: {p_value:.6f}")
print(f"Significance level (alpha): {alpha}")

print("\nDecision:")
if p_value < alpha:
    print("Reject H0")
    print("There is statistically significant evidence of a")
    print("linear relationship between the two variables.")
else:
    print("Fail to reject H0")
    print("There is insufficient statistical evidence of a")
    print("linear relationship between the two variables.")

HYPOTHESIS TEST 1 — PEARSON CORRELATION

Research Question:
Is there a statistically significant linear relationship
between DC ATM Withdrawal Volume and NPCI Volume?

H0:
There is no statistically significant linear relationship.

H1:
There is a statistically significant linear relationship.

Results:
Pearson correlation (r): -0.9069
P-value: 0.093085
Significance level (alpha): 0.05

Decision:
Fail to reject H0
There is insufficient statistical evidence of a
linear relationship between the two variables.


In [6]:
import pandas as pd
from scipy.stats import pearsonr

# ============================================================
# HYPOTHESIS TEST 2 — DC ATM WITHDRAWAL vs NPCI VALUE
# ============================================================

# Load the dataset
df = pd.read_csv("ml_prepared_2025.csv")

# ------------------------------------------------------------
# Create quarterly data
# ------------------------------------------------------------

quarterly = df.groupby("Quarter").agg({
    "DC_ATM_Withdrawal_Volume": "sum",
    "NPCI_Value_Bn": "first"
}).reset_index()

print("QUARTERLY DATA FOR HYPOTHESIS TEST 2")
print("=" * 60)
print(quarterly.to_string(index=False))

print("\nNumber of quarterly observations:", len(quarterly))

# ------------------------------------------------------------
# Check missing values
# ------------------------------------------------------------

print("\nMissing values:")
print(
    quarterly[
        ["DC_ATM_Withdrawal_Volume", "NPCI_Value_Bn"]
    ].isnull().sum()
)

# ------------------------------------------------------------
# Pearson correlation
# ------------------------------------------------------------

x = quarterly["DC_ATM_Withdrawal_Volume"]
y = quarterly["NPCI_Value_Bn"]

r, p_value = pearsonr(x, y)

alpha = 0.05

print("\n")
print("HYPOTHESIS TEST 2 — PEARSON CORRELATION")
print("=" * 60)

print("\nResearch Question:")
print(
    "Is there a statistically significant linear relationship "
    "between DC ATM Withdrawal Volume and NPCI Value?"
)

print("\nH0:")
print(
    "There is no statistically significant linear relationship "
    "between DC ATM Withdrawal Volume and NPCI Value."
)

print("\nH1:")
print(
    "There is a statistically significant linear relationship "
    "between DC ATM Withdrawal Volume and NPCI Value."
)

print("\nResults:")
print(f"Pearson correlation (r): {r:.4f}")
print(f"P-value: {p_value:.6f}")
print(f"Significance level (alpha): {alpha}")

# ------------------------------------------------------------
# Decision
# ------------------------------------------------------------

print("\nDecision:")

if p_value < alpha:
    print("Reject H0")
    print(
        "There is statistically significant evidence of "
        "a linear relationship between the two variables."
    )
else:
    print("Fail to reject H0")
    print(
        "There is insufficient statistical evidence of "
        "a linear relationship between the two variables."
    )

QUARTERLY DATA FOR HYPOTHESIS TEST 2
Quarter  DC_ATM_Withdrawal_Volume  NPCI_Value_Bn
     Q1                1431451650  224110.140640
     Q2                1351651681  226765.475487
     Q3                1341992348  245373.027942
     Q4                1346057982  255516.058420

Number of quarterly observations: 4

Missing values:
DC_ATM_Withdrawal_Volume    0
NPCI_Value_Bn               0
dtype: int64


HYPOTHESIS TEST 2 — PEARSON CORRELATION

Research Question:
Is there a statistically significant linear relationship between DC ATM Withdrawal Volume and NPCI Value?

H0:
There is no statistically significant linear relationship between DC ATM Withdrawal Volume and NPCI Value.

H1:
There is a statistically significant linear relationship between DC ATM Withdrawal Volume and NPCI Value.

Results:
Pearson correlation (r): -0.6615
P-value: 0.338524
Significance level (alpha): 0.05

Decision:
Fail to reject H0
There is insufficient statistical evidence of a linear relationship between t

In [7]:
import pandas as pd
from scipy.stats import pearsonr

# ============================================================
# HYPOTHESIS TEST 3 — POS vs NPCI VOLUME
# ============================================================

df = pd.read_csv("ml_prepared_2025.csv")

# Create quarterly data
quarterly = df.groupby("Quarter").agg({
    "POS": "sum",
    "NPCI_Volume_Mn": "first"
}).reset_index()

print("QUARTERLY DATA FOR HYPOTHESIS TEST 3")
print("=" * 60)
print(quarterly.to_string(index=False))

print("\nNumber of quarterly observations:", len(quarterly))

# Check missing values
print("\nMissing values:")
print(
    quarterly[
        ["POS", "NPCI_Volume_Mn"]
    ].isnull().sum()
)

# Pearson correlation
x = quarterly["POS"]
y = quarterly["NPCI_Volume_Mn"]

r, p_value = pearsonr(x, y)

alpha = 0.05

print("\n")
print("HYPOTHESIS TEST 3 — PEARSON CORRELATION")
print("=" * 60)

print("\nResearch Question:")
print(
    "Is there a statistically significant linear relationship "
    "between POS and NPCI Volume?"
)

print("\nH0:")
print(
    "There is no statistically significant linear relationship "
    "between POS and NPCI Volume."
)

print("\nH1:")
print(
    "There is a statistically significant linear relationship "
    "between POS and NPCI Volume."
)

print("\nResults:")
print(f"Pearson correlation (r): {r:.4f}")
print(f"P-value: {p_value:.6f}")
print(f"Significance level (alpha): {alpha}")

print("\nDecision:")

if p_value < alpha:
    print("Reject H0")
    print(
        "There is statistically significant evidence of "
        "a linear relationship between POS and NPCI Volume."
    )
else:
    print("Fail to reject H0")
    print(
        "There is insufficient statistical evidence of "
        "a linear relationship between POS and NPCI Volume."
    )

QUARTERLY DATA FOR HYPOTHESIS TEST 3
Quarter      POS  NPCI_Volume_Mn
     Q1 32168295   132106.807145
     Q2 34647381   141651.024411
     Q3 35993795   148237.297964
     Q4 35036338   151451.826569

Number of quarterly observations: 4

Missing values:
POS               0
NPCI_Volume_Mn    0
dtype: int64


HYPOTHESIS TEST 3 — PEARSON CORRELATION

Research Question:
Is there a statistically significant linear relationship between POS and NPCI Volume?

H0:
There is no statistically significant linear relationship between POS and NPCI Volume.

H1:
There is a statistically significant linear relationship between POS and NPCI Volume.

Results:
Pearson correlation (r): 0.9006
P-value: 0.099425
Significance level (alpha): 0.05

Decision:
Fail to reject H0
There is insufficient statistical evidence of a linear relationship between POS and NPCI Volume.


In [8]:
import pandas as pd
from scipy.stats import pearsonr

# ============================================================
# HYPOTHESIS TEST 4 — POS vs NPCI VALUE
# ============================================================

df = pd.read_csv("ml_prepared_2025.csv")

# Create quarterly data
quarterly = df.groupby("Quarter").agg({
    "POS": "sum",
    "NPCI_Value_Bn": "first"
}).reset_index()

print("QUARTERLY DATA FOR HYPOTHESIS TEST 4")
print("=" * 60)
print(quarterly.to_string(index=False))

print("\nNumber of quarterly observations:", len(quarterly))

# Check missing values
print("\nMissing values:")
print(
    quarterly[
        ["POS", "NPCI_Value_Bn"]
    ].isnull().sum()
)

# Pearson correlation
x = quarterly["POS"]
y = quarterly["NPCI_Value_Bn"]

r, p_value = pearsonr(x, y)

alpha = 0.05

print("\n")
print("HYPOTHESIS TEST 4 — PEARSON CORRELATION")
print("=" * 60)

print("\nResearch Question:")
print(
    "Is there a statistically significant linear relationship "
    "between POS and NPCI Value?"
)

print("\nH0:")
print(
    "There is no statistically significant linear relationship "
    "between POS and NPCI Value."
)

print("\nH1:")
print(
    "There is a statistically significant linear relationship "
    "between POS and NPCI Value."
)

print("\nResults:")
print(f"Pearson correlation (r): {r:.4f}")
print(f"P-value: {p_value:.6f}")
print(f"Significance level (alpha): {alpha}")

print("\nDecision:")

if p_value < alpha:
    print("Reject H0")
    print(
        "There is statistically significant evidence of "
        "a linear relationship between POS and NPCI Value."
    )
else:
    print("Fail to reject H0")
    print(
        "There is insufficient statistical evidence of "
        "a linear relationship between POS and NPCI Value."
    )

QUARTERLY DATA FOR HYPOTHESIS TEST 4
Quarter      POS  NPCI_Value_Bn
     Q1 32168295  224110.140640
     Q2 34647381  226765.475487
     Q3 35993795  245373.027942
     Q4 35036338  255516.058420

Number of quarterly observations: 4

Missing values:
POS              0
NPCI_Value_Bn    0
dtype: int64


HYPOTHESIS TEST 4 — PEARSON CORRELATION

Research Question:
Is there a statistically significant linear relationship between POS and NPCI Value?

H0:
There is no statistically significant linear relationship between POS and NPCI Value.

H1:
There is a statistically significant linear relationship between POS and NPCI Value.

Results:
Pearson correlation (r): 0.6943
P-value: 0.305690
Significance level (alpha): 0.05

Decision:
Fail to reject H0
There is insufficient statistical evidence of a linear relationship between POS and NPCI Value.


In [9]:
import pandas as pd
from scipy.stats import pearsonr

# ============================================================
# HYPOTHESIS TEST 5 — MICRO ATM vs NPCI VOLUME
# ============================================================

df = pd.read_csv("ml_prepared_2025.csv")

# Create quarterly data
quarterly = df.groupby("Quarter").agg({
    "Micro_ATM": "sum",
    "NPCI_Volume_Mn": "first"
}).reset_index()

print("QUARTERLY DATA FOR HYPOTHESIS TEST 5")
print("=" * 60)
print(quarterly.to_string(index=False))

print("\nNumber of quarterly observations:", len(quarterly))

# Check missing values
print("\nMissing values:")
print(
    quarterly[
        ["Micro_ATM", "NPCI_Volume_Mn"]
    ].isnull().sum()
)

# Pearson correlation
x = quarterly["Micro_ATM"]
y = quarterly["NPCI_Volume_Mn"]

r, p_value = pearsonr(x, y)

alpha = 0.05

print("\n")
print("HYPOTHESIS TEST 5 — PEARSON CORRELATION")
print("=" * 60)

print("\nResearch Question:")
print(
    "Is there a statistically significant linear relationship "
    "between Micro ATM and NPCI Volume?"
)

print("\nH0:")
print(
    "There is no statistically significant linear relationship "
    "between Micro ATM and NPCI Volume."
)

print("\nH1:")
print(
    "There is a statistically significant linear relationship "
    "between Micro ATM and NPCI Volume."
)

print("\nResults:")
print(f"Pearson correlation (r): {r:.4f}")
print(f"P-value: {p_value:.6f}")
print(f"Significance level (alpha): {alpha}")

print("\nDecision:")

if p_value < alpha:
    print("Reject H0")
    print(
        "There is statistically significant evidence of "
        "a linear relationship between Micro ATM and NPCI Volume."
    )
else:
    print("Fail to reject H0")
    print(
        "There is insufficient statistical evidence of "
        "a linear relationship between Micro ATM and NPCI Volume."
    )

QUARTERLY DATA FOR HYPOTHESIS TEST 5
Quarter  Micro_ATM  NPCI_Volume_Mn
     Q1    4423748   132106.807145
     Q2    4410929   141651.024411
     Q3    4398035   148237.297964
     Q4    4306306   151451.826569

Number of quarterly observations: 4

Missing values:
Micro_ATM         0
NPCI_Volume_Mn    0
dtype: int64


HYPOTHESIS TEST 5 — PEARSON CORRELATION

Research Question:
Is there a statistically significant linear relationship between Micro ATM and NPCI Volume?

H0:
There is no statistically significant linear relationship between Micro ATM and NPCI Volume.

H1:
There is a statistically significant linear relationship between Micro ATM and NPCI Volume.

Results:
Pearson correlation (r): -0.7709
P-value: 0.229130
Significance level (alpha): 0.05

Decision:
Fail to reject H0
There is insufficient statistical evidence of a linear relationship between Micro ATM and NPCI Volume.


In [12]:
# ============================================================
# HYPOTHESIS TEST 6 — MICRO ATM vs NPCI VALUE
# ============================================================

import pandas as pd
from scipy.stats import pearsonr

# ------------------------------------------------------------
# STEP 1: CREATE VERIFIED QUARTERLY DATA
# ------------------------------------------------------------

df_test6 = pd.DataFrame({
    "Quarter": ["Q1", "Q2", "Q3", "Q4"],
    "Micro_ATM": [
        4423748,
        4410929,
        4398035,
        4306306
    ],
    "NPCI_Value_Bn": [
        224110.140640,
        226765.475487,
        245373.027942,
        255516.058420
    ]
})

print("QUARTERLY DATA FOR HYPOTHESIS TEST 6")
print("=" * 60)

print(df_test6.to_string(index=False))

print("\nNumber of quarterly observations:", len(df_test6))


# ------------------------------------------------------------
# STEP 2: CHECK MISSING VALUES
# ------------------------------------------------------------

print("\nMissing values:")
print(
    df_test6[
        ["Micro_ATM", "NPCI_Value_Bn"]
    ].isnull().sum()
)


# ------------------------------------------------------------
# STEP 3: DESCRIPTIVE STATISTICS
# ------------------------------------------------------------

print("\nDescriptive Statistics:")
print(
    df_test6[
        ["Micro_ATM", "NPCI_Value_Bn"]
    ].describe()
)


# ------------------------------------------------------------
# STEP 4: DEFINE VARIABLES
# ------------------------------------------------------------

x = df_test6["Micro_ATM"]
y = df_test6["NPCI_Value_Bn"]


# ------------------------------------------------------------
# STEP 5: PEARSON CORRELATION
# ------------------------------------------------------------

r, p_value = pearsonr(x, y)

alpha = 0.05


# ------------------------------------------------------------
# STEP 6: DISPLAY HYPOTHESIS
# ------------------------------------------------------------

print("\n")
print("HYPOTHESIS TEST 6 — PEARSON CORRELATION")
print("=" * 60)

print("\nResearch Question:")
print(
    "Is there a statistically significant linear relationship "
    "between Micro ATM and NPCI Value?"
)

print("\nH0:")
print(
    "There is no statistically significant linear relationship "
    "between Micro ATM and NPCI Value."
)

print("\nH1:")
print(
    "There is a statistically significant linear relationship "
    "between Micro ATM and NPCI Value."
)


# ------------------------------------------------------------
# STEP 7: RESULTS
# ------------------------------------------------------------

print("\nResults:")
print(f"Pearson correlation (r): {r:.4f}")
print(f"P-value: {p_value:.6f}")
print(f"Significance level (alpha): {alpha}")


# ------------------------------------------------------------
# STEP 8: DECISION
# ------------------------------------------------------------

print("\nDecision:")

if p_value < alpha:
    print("Reject H0")
    print(
        "There is statistically significant evidence of "
        "a linear relationship between Micro ATM and NPCI Value."
    )
else:
    print("Fail to reject H0")
    print(
        "There is insufficient statistical evidence of "
        "a linear relationship between Micro ATM and NPCI Value."
    )


# ------------------------------------------------------------
# STEP 9: INTERPRETATION OF CORRELATION
# ------------------------------------------------------------

print("\nCorrelation Interpretation:")

if r > 0:
    print("The relationship is positive.")
elif r < 0:
    print("The relationship is negative.")
else:
    print("There is no linear correlation.")

print(f"Correlation coefficient: {r:.4f}")


# ------------------------------------------------------------
# STEP 10: FINAL SUMMARY
# ------------------------------------------------------------

print("\n")
print("=" * 60)
print("HYPOTHESIS TEST 6 — FINAL SUMMARY")
print("=" * 60)

print(f"Variable 1 : Micro_ATM")
print(f"Variable 2 : NPCI_Value_Bn")
print(f"Observations : {len(df_test6)}")
print(f"Pearson r : {r:.4f}")
print(f"P-value : {p_value:.6f}")
print(f"Alpha : {alpha}")

if p_value < alpha:
    print("Result : Statistically significant")
    print("Decision : Reject H0")
else:
    print("Result : Not statistically significant")
    print("Decision : Fail to reject H0")

print("=" * 60)

QUARTERLY DATA FOR HYPOTHESIS TEST 6
Quarter  Micro_ATM  NPCI_Value_Bn
     Q1    4423748  224110.140640
     Q2    4410929  226765.475487
     Q3    4398035  245373.027942
     Q4    4306306  255516.058420

Number of quarterly observations: 4

Missing values:
Micro_ATM        0
NPCI_Value_Bn    0
dtype: int64

Descriptive Statistics:
          Micro_ATM  NPCI_Value_Bn
count  4.000000e+00       4.000000
mean   4.384754e+06  237941.175622
std    5.334209e+04   15058.804030
min    4.306306e+06  224110.140640
25%    4.375103e+06  226101.641775
50%    4.404482e+06  236069.251714
75%    4.414134e+06  247908.785562
max    4.423748e+06  255516.058420


HYPOTHESIS TEST 6 — PEARSON CORRELATION

Research Question:
Is there a statistically significant linear relationship between Micro ATM and NPCI Value?

H0:
There is no statistically significant linear relationship between Micro ATM and NPCI Value.

H1:
There is a statistically significant linear relationship between Micro ATM and NPCI Value.

R

In [14]:
# ============================================================
# CHECK DC ONLINE VOLUME BEFORE HYPOTHESIS TEST 7
# ============================================================

print("DC_Online_Volume CHECK")
print("=" * 60)

print("\nUnique values:")
print(df["DC_Online_Volume"].unique())

print("\nNumber of unique values:")
print(df["DC_Online_Volume"].nunique())

print("\nMonthly values:")
print(
    df[
        ["Month", "Quarter", "DC_Online_Volume"]
    ].to_string(index=False)
)

print("\nDescriptive statistics:")
print(df["DC_Online_Volume"].describe())

DC_Online_Volume CHECK

Unique values:
[30470254 27813305 30622450 29478270 28805524 27210465 28764405 27641922
 26904171 26666791 24803398 25407654]

Number of unique values:
12

Monthly values:
    Month Quarter  DC_Online_Volume
  January      Q1          30470254
 February      Q1          27813305
    March      Q1          30622450
    April      Q2          29478270
      May      Q2          28805524
     June      Q2          27210465
     July      Q3          28764405
   August      Q3          27641922
September      Q3          26904171
  October      Q4          26666791
 November      Q4          24803398
 December      Q4          25407654

Descriptive statistics:
count    1.200000e+01
mean     2.788238e+07
std      1.832413e+06
min      2.480340e+07
25%      2.684483e+07
50%      2.772761e+07
75%      2.897371e+07
max      3.062245e+07
Name: DC_Online_Volume, dtype: float64


In [16]:
# ============================================================
# HYPOTHESIS TEST 7
# DC Online Volume vs NPCI Volume
# ============================================================

import pandas as pd
import numpy as np
from scipy.stats import pearsonr
from pathlib import Path

# ============================================================
# 1. FIND THE CORRECT DATASET
# ============================================================

analytics_path = Path(r"C:\Users\THARUNI REDDY V\Downloads\analytics")

required_columns = [
    "Quarter",
    "DC_Online_Volume",
    "NPCI_Volume_Mn"
]

csv_files = list(analytics_path.glob("*.csv"))

print("CSV files found:", len(csv_files))

selected_file = None

for file in csv_files:
    try:
        temp = pd.read_csv(file)

        if all(col in temp.columns for col in required_columns):
            selected_file = file
            print("\nCorrect dataset found:")
            print(file)
            break

    except Exception:
        continue

if selected_file is None:
    raise FileNotFoundError(
        "No CSV file containing Quarter, DC_Online_Volume "
        "and NPCI_Volume_Mn was found."
    )

# Load dataset
df = pd.read_csv(selected_file)

print("\nDataset shape:", df.shape)

# ============================================================
# 2. CHECK REQUIRED COLUMNS
# ============================================================

print("\nRequired columns:")
print(required_columns)

print("\nAvailable columns:")
print(df.columns.tolist())

# ============================================================
# 3. CHECK DC ONLINE VOLUME
# ============================================================

print("\n============================================================")
print("DC ONLINE VOLUME CHECK")
print("============================================================")

print("\nUnique values:")
print(df["DC_Online_Volume"].unique())

print("\nNumber of unique values:")
print(df["DC_Online_Volume"].nunique())

print("\nMissing values:")
print(
    df[
        ["DC_Online_Volume", "NPCI_Volume_Mn"]
    ].isnull().sum()
)

# ============================================================
# 4. CONVERT VARIABLES TO NUMERIC
# ============================================================

df["DC_Online_Volume"] = pd.to_numeric(
    df["DC_Online_Volume"],
    errors="coerce"
)

df["NPCI_Volume_Mn"] = pd.to_numeric(
    df["NPCI_Volume_Mn"],
    errors="coerce"
)

# ============================================================
# 5. CREATE QUARTERLY DATA
# ============================================================

df_quarterly = (
    df.groupby("Quarter", sort=False)
      .agg({
          "DC_Online_Volume": "sum",
          "NPCI_Volume_Mn": "first"
      })
      .reset_index()
)

print("\n============================================================")
print("QUARTERLY DATA FOR HYPOTHESIS TEST 7")
print("============================================================")

print(
    df_quarterly.to_string(index=False)
)

print(
    "\nNumber of quarterly observations:",
    len(df_quarterly)
)

# ============================================================
# 6. CHECK FOR ZERO / CONSTANT VALUES
# ============================================================

print("\n============================================================")
print("QUARTERLY VARIABLE CHECK")
print("============================================================")

print(
    "\nDC_Online_Volume unique quarterly values:",
    df_quarterly["DC_Online_Volume"].nunique()
)

print(
    "NPCI_Volume_Mn unique quarterly values:",
    df_quarterly["NPCI_Volume_Mn"].nunique()
)

# ============================================================
# 7. DESCRIPTIVE STATISTICS
# ============================================================

print("\n============================================================")
print("DESCRIPTIVE STATISTICS")
print("============================================================")

print(
    df_quarterly[
        ["DC_Online_Volume", "NPCI_Volume_Mn"]
    ].describe()
)

# ============================================================
# 8. HYPOTHESIS TEST
# ============================================================

print("\n============================================================")
print("HYPOTHESIS TEST 7 — PEARSON CORRELATION")
print("============================================================")

print("\nResearch Question:")
print(
    "Is there a statistically significant linear relationship "
    "between DC Online Volume and NPCI Volume?"
)

print("\nH0:")
print(
    "There is no statistically significant linear relationship "
    "between DC Online Volume and NPCI Volume."
)

print("\nH1:")
print(
    "There is a statistically significant linear relationship "
    "between DC Online Volume and NPCI Volume."
)

alpha = 0.05

x = df_quarterly["DC_Online_Volume"]
y = df_quarterly["NPCI_Volume_Mn"]

# Check constant input before Pearson correlation
if x.nunique() <= 1 or y.nunique() <= 1:

    print("\nERROR:")
    print(
        "Pearson correlation cannot be calculated because "
        "one variable is constant."
    )

else:

    r, p_value = pearsonr(x, y)

    print("\nResults:")
    print(f"Pearson correlation (r): {r:.4f}")
    print(f"P-value: {p_value:.6f}")
    print(f"Significance level (alpha): {alpha}")

    # ========================================================
    # 9. DECISION
    # ========================================================

    if p_value < alpha:

        decision = "Reject H0"
        result = "Statistically significant"

        print("\nDecision:")
        print("Reject H0")

        print(
            "There is statistically significant evidence of "
            "a linear relationship between DC Online Volume "
            "and NPCI Volume."
        )

    else:

        decision = "Fail to reject H0"
        result = "Not statistically significant"

        print("\nDecision:")
        print("Fail to reject H0")

        print(
            "There is insufficient statistical evidence of "
            "a linear relationship between DC Online Volume "
            "and NPCI Volume."
        )

    # ========================================================
    # 10. CORRELATION INTERPRETATION
    # ========================================================

    print("\nCorrelation Interpretation:")

    if r > 0:
        direction = "Positive"
        print("The relationship is positive.")

    elif r < 0:
        direction = "Negative"
        print("The relationship is negative.")

    else:
        direction = "No linear relationship"
        print("There is no linear correlation.")

    print(f"Correlation coefficient: {r:.4f}")

    # ========================================================
    # 11. FINAL SUMMARY
    # ========================================================

    print("\n")
    print("============================================================")
    print("HYPOTHESIS TEST 7 — FINAL SUMMARY")
    print("============================================================")

    print("Variable 1 :", "DC_Online_Volume")
    print("Variable 2 :", "NPCI_Volume_Mn")
    print("Observations :", len(df_quarterly))
    print(f"Pearson r : {r:.4f}")
    print(f"P-value : {p_value:.6f}")
    print(f"Alpha : {alpha}")
    print(f"Result : {result}")
    print(f"Decision : {decision}")

    print("============================================================")

CSV files found: 67

Correct dataset found:
C:\Users\THARUNI REDDY V\Downloads\analytics\ml_prepared_2025.csv

Dataset shape: (12, 40)

Required columns:
['Quarter', 'DC_Online_Volume', 'NPCI_Volume_Mn']

Available columns:
['Year', 'Month_Num', 'Month', 'Quarter', 'ATM_Onsite', 'ATM_Offsite', 'POS', 'Micro_ATM', 'Bharat_QR', 'UPI_QR', 'Credit_Cards', 'Debit_Cards', 'CC_POS_Volume', 'CC_Online_Volume', 'DC_POS_Volume', 'DC_Online_Volume', 'DC_ATM_Withdrawal_Volume', 'PSI_Volume_Current_Month', 'PSI_Volume_Previous_Month', 'PSI_Volume_Same_Month_Last_Year', 'PSI_Value_Current_Month', 'PSI_Value_Previous_Month', 'PSI_Value_Same_Month_Last_Year', 'Month_Sin', 'Month_Cos', 'Time_Index', 'Total_Card_Payment_Volume', 'CreditCard_Payment_Volume', 'DebitCard_Payment_Volume', 'Total_QR_Infrastructure', 'Total_ATM_Infrastructure', 'Total_Digital_Infrastructure', 'Credit_Debit_Card_Ratio', 'CC_to_DC_Payment_Volume_Ratio', 'PSI_Volume_Gap_Current_Previous', 'PSI_Value_Gap_Current_Previous', 'PSI_V

In [17]:
# ============================================================
# HYPOTHESIS TEST 8
# UPI QR vs NPCI Volume
# Pearson Correlation
# ============================================================

import pandas as pd
from scipy.stats import pearsonr

# ------------------------------------------------------------
# 1. LOAD THE CORRECT DATASET
# ------------------------------------------------------------

file_path = r"C:\Users\THARUNI REDDY V\Downloads\analytics\ml_prepared_2025.csv"

df = pd.read_csv(file_path)

print("Dataset loaded successfully")
print("Dataset shape:", df.shape)


# ------------------------------------------------------------
# 2. CHECK REQUIRED COLUMNS
# ------------------------------------------------------------

required_columns = [
    "Quarter",
    "UPI_QR",
    "NPCI_Volume_Mn"
]

missing_columns = [
    col for col in required_columns
    if col not in df.columns
]

if missing_columns:
    raise ValueError(
        f"Missing required columns: {missing_columns}"
    )

print("\nRequired columns found:")
print(required_columns)


# ------------------------------------------------------------
# 3. CHECK MONTHLY DATA
# ------------------------------------------------------------

print("\n============================================================")
print("UPI QR CHECK")
print("============================================================")

print("\nUnique values:")
print(df["UPI_QR"].unique())

print("\nNumber of unique values:")
print(df["UPI_QR"].nunique())

print("\nMissing values:")
print(
    df[["UPI_QR", "NPCI_Volume_Mn"]].isnull().sum()
)


# ------------------------------------------------------------
# 4. CREATE QUARTERLY DATA
# ------------------------------------------------------------

df_quarterly = (
    df.groupby("Quarter", as_index=False)
      .agg({
          "UPI_QR": "sum",
          "NPCI_Volume_Mn": "first"
      })
)

# Ensure correct quarter order
quarter_order = ["Q1", "Q2", "Q3", "Q4"]

df_quarterly["Quarter"] = pd.Categorical(
    df_quarterly["Quarter"],
    categories=quarter_order,
    ordered=True
)

df_quarterly = (
    df_quarterly
    .sort_values("Quarter")
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 5. DISPLAY QUARTERLY DATA
# ------------------------------------------------------------

print("\n============================================================")
print("QUARTERLY DATA FOR HYPOTHESIS TEST 8")
print("============================================================")

print(
    df_quarterly[
        ["Quarter", "UPI_QR", "NPCI_Volume_Mn"]
    ].to_string(index=False)
)

print(
    "\nNumber of quarterly observations:",
    len(df_quarterly)
)


# ------------------------------------------------------------
# 6. VARIABLE CHECK
# ------------------------------------------------------------

print("\n============================================================")
print("QUARTERLY VARIABLE CHECK")
print("============================================================")

print(
    "UPI_QR unique quarterly values:",
    df_quarterly["UPI_QR"].nunique()
)

print(
    "NPCI_Volume_Mn unique quarterly values:",
    df_quarterly["NPCI_Volume_Mn"].nunique()
)


# ------------------------------------------------------------
# 7. MISSING VALUE CHECK
# ------------------------------------------------------------

print("\nMissing values:")

print(
    df_quarterly[
        ["UPI_QR", "NPCI_Volume_Mn"]
    ].isnull().sum()
)


# ------------------------------------------------------------
# 8. DESCRIPTIVE STATISTICS
# ------------------------------------------------------------

print("\n============================================================")
print("DESCRIPTIVE STATISTICS")
print("============================================================")

print(
    df_quarterly[
        ["UPI_QR", "NPCI_Volume_Mn"]
    ].describe()
)


# ------------------------------------------------------------
# 9. PEARSON CORRELATION
# ------------------------------------------------------------

x = df_quarterly["UPI_QR"]
y = df_quarterly["NPCI_Volume_Mn"]

# Check for constant variables
if x.nunique() < 2 or y.nunique() < 2:
    print("\nPearson correlation cannot be calculated.")
    print("One of the variables is constant.")
else:

    r, p_value = pearsonr(x, y)

    alpha = 0.05

    print("\n============================================================")
    print("HYPOTHESIS TEST 8 — PEARSON CORRELATION")
    print("============================================================")

    print("\nResearch Question:")
    print(
        "Is there a statistically significant linear relationship "
        "between UPI QR and NPCI Volume?"
    )

    print("\nH0:")
    print(
        "There is no statistically significant linear relationship "
        "between UPI QR and NPCI Volume."
    )

    print("\nH1:")
    print(
        "There is a statistically significant linear relationship "
        "between UPI QR and NPCI Volume."
    )

    print("\nResults:")
    print(f"Pearson correlation (r): {r:.4f}")
    print(f"P-value: {p_value:.6f}")
    print(f"Significance level (alpha): {alpha}")

    # --------------------------------------------------------
    # 10. DECISION
    # --------------------------------------------------------

    if p_value < alpha:
        decision = "Reject H0"
        result = "Statistically significant"
        conclusion = (
            "There is sufficient statistical evidence of "
            "a linear relationship between UPI QR and NPCI Volume."
        )
    else:
        decision = "Fail to reject H0"
        result = "Not statistically significant"
        conclusion = (
            "There is insufficient statistical evidence of "
            "a linear relationship between UPI QR and NPCI Volume."
        )

    print("\nDecision:")
    print(decision)

    print(conclusion)

    # --------------------------------------------------------
    # 11. CORRELATION INTERPRETATION
    # --------------------------------------------------------

    print("\nCorrelation Interpretation:")

    if r > 0:
        print("The relationship is positive.")
    elif r < 0:
        print("The relationship is negative.")
    else:
        print("There is no linear correlation.")

    print(f"Correlation coefficient: {r:.4f}")


    # --------------------------------------------------------
    # 12. FINAL SUMMARY
    # --------------------------------------------------------

    print("\n")
    print("============================================================")
    print("HYPOTHESIS TEST 8 — FINAL SUMMARY")
    print("============================================================")

    print("Variable 1 : UPI_QR")
    print("Variable 2 : NPCI_Volume_Mn")
    print("Observations :", len(df_quarterly))
    print(f"Pearson r : {r:.4f}")
    print(f"P-value : {p_value:.6f}")
    print(f"Alpha : {alpha}")
    print(f"Result : {result}")
    print(f"Decision : {decision}")

    print("============================================================")

Dataset loaded successfully
Dataset shape: (12, 40)

Required columns found:
['Quarter', 'UPI_QR', 'NPCI_Volume_Mn']

UPI QR CHECK

Unique values:
[6.40164702e+08 6.49691280e+08 6.57929517e+08 6.62384943e+08
 6.69729906e+08 6.78160811e+08 6.88030939e+08 6.97747231e+08
 7.08974740e+08 7.17433868e+08 7.28199556e+08 7.31365285e+08]

Number of unique values:
12

Missing values:
UPI_QR            0
NPCI_Volume_Mn    0
dtype: int64

QUARTERLY DATA FOR HYPOTHESIS TEST 8
Quarter       UPI_QR  NPCI_Volume_Mn
     Q1 1947785499.0   132106.807145
     Q2 2010275660.0   141651.024411
     Q3 2094752910.0   148237.297964
     Q4 2176998708.5   151451.826569

Number of quarterly observations: 4

QUARTERLY VARIABLE CHECK
UPI_QR unique quarterly values: 4
NPCI_Volume_Mn unique quarterly values: 4

Missing values:
UPI_QR            0
NPCI_Volume_Mn    0
dtype: int64

DESCRIPTIVE STATISTICS
             UPI_QR  NPCI_Volume_Mn
count  4.000000e+00        4.000000
mean   2.057453e+09   143361.739022
std   

In [18]:
# ============================================================
# HYPOTHESIS TEST 9
# Bharat QR vs NPCI Volume
# Pearson Correlation
# ============================================================

import pandas as pd
from scipy.stats import pearsonr

# ------------------------------------------------------------
# 1. LOAD CORRECT DATASET
# ------------------------------------------------------------

file_path = r"C:\Users\THARUNI REDDY V\Downloads\analytics\ml_prepared_2025.csv"

df = pd.read_csv(file_path)

print("Dataset loaded successfully")
print("Dataset shape:", df.shape)


# ------------------------------------------------------------
# 2. CHECK REQUIRED COLUMNS
# ------------------------------------------------------------

required_columns = [
    "Quarter",
    "Bharat_QR",
    "NPCI_Volume_Mn"
]

missing_columns = [
    col for col in required_columns
    if col not in df.columns
]

if missing_columns:
    raise ValueError(
        f"Missing required columns: {missing_columns}"
    )

print("\nRequired columns found:")
print(required_columns)


# ------------------------------------------------------------
# 3. MONTHLY VARIABLE CHECK
# ------------------------------------------------------------

print("\n============================================================")
print("BHARAT QR CHECK")
print("============================================================")

print("\nUnique values:")
print(df["Bharat_QR"].unique())

print("\nNumber of unique values:")
print(df["Bharat_QR"].nunique())

print("\nMissing values:")
print(
    df[["Bharat_QR", "NPCI_Volume_Mn"]].isnull().sum()
)


# ------------------------------------------------------------
# 4. CREATE QUARTERLY DATA
# ------------------------------------------------------------

df_quarterly = (
    df.groupby("Quarter", as_index=False)
      .agg({
          "Bharat_QR": "sum",
          "NPCI_Volume_Mn": "first"
      })
)

# Correct quarter ordering
quarter_order = ["Q1", "Q2", "Q3", "Q4"]

df_quarterly["Quarter"] = pd.Categorical(
    df_quarterly["Quarter"],
    categories=quarter_order,
    ordered=True
)

df_quarterly = (
    df_quarterly
    .sort_values("Quarter")
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 5. DISPLAY QUARTERLY DATA
# ------------------------------------------------------------

print("\n============================================================")
print("QUARTERLY DATA FOR HYPOTHESIS TEST 9")
print("============================================================")

print(
    df_quarterly[
        ["Quarter", "Bharat_QR", "NPCI_Volume_Mn"]
    ].to_string(index=False)
)

print(
    "\nNumber of quarterly observations:",
    len(df_quarterly)
)


# ------------------------------------------------------------
# 6. VARIABLE CHECK
# ------------------------------------------------------------

print("\n============================================================")
print("QUARTERLY VARIABLE CHECK")
print("============================================================")

print(
    "Bharat_QR unique quarterly values:",
    df_quarterly["Bharat_QR"].nunique()
)

print(
    "NPCI_Volume_Mn unique quarterly values:",
    df_quarterly["NPCI_Volume_Mn"].nunique()
)


# ------------------------------------------------------------
# 7. MISSING VALUE CHECK
# ------------------------------------------------------------

print("\nMissing values:")

print(
    df_quarterly[
        ["Bharat_QR", "NPCI_Volume_Mn"]
    ].isnull().sum()
)


# ------------------------------------------------------------
# 8. DESCRIPTIVE STATISTICS
# ------------------------------------------------------------

print("\n============================================================")
print("DESCRIPTIVE STATISTICS")
print("============================================================")

print(
    df_quarterly[
        ["Bharat_QR", "NPCI_Volume_Mn"]
    ].describe()
)


# ------------------------------------------------------------
# 9. PEARSON CORRELATION
# ------------------------------------------------------------

x = df_quarterly["Bharat_QR"]
y = df_quarterly["NPCI_Volume_Mn"]

# Prevent Pearson calculation if variable is constant
if x.nunique() < 2 or y.nunique() < 2:

    print("\nPearson correlation cannot be calculated.")
    print("One of the variables is constant.")

else:

    r, p_value = pearsonr(x, y)

    alpha = 0.05

    print("\n============================================================")
    print("HYPOTHESIS TEST 9 — PEARSON CORRELATION")
    print("============================================================")

    print("\nResearch Question:")
    print(
        "Is there a statistically significant linear relationship "
        "between Bharat QR and NPCI Volume?"
    )

    print("\nH0:")
    print(
        "There is no statistically significant linear relationship "
        "between Bharat QR and NPCI Volume."
    )

    print("\nH1:")
    print(
        "There is a statistically significant linear relationship "
        "between Bharat QR and NPCI Volume."
    )

    print("\nResults:")
    print(f"Pearson correlation (r): {r:.4f}")
    print(f"P-value: {p_value:.6f}")
    print(f"Significance level (alpha): {alpha}")


    # --------------------------------------------------------
    # 10. STATISTICAL DECISION
    # --------------------------------------------------------

    if p_value < alpha:

        decision = "Reject H0"
        result = "Statistically significant"

        conclusion = (
            "There is sufficient statistical evidence of "
            "a linear relationship between Bharat QR and NPCI Volume."
        )

    else:

        decision = "Fail to reject H0"
        result = "Not statistically significant"

        conclusion = (
            "There is insufficient statistical evidence of "
            "a linear relationship between Bharat QR and NPCI Volume."
        )

    print("\nDecision:")
    print(decision)

    print(conclusion)


    # --------------------------------------------------------
    # 11. CORRELATION INTERPRETATION
    # --------------------------------------------------------

    print("\nCorrelation Interpretation:")

    if r > 0:
        print("The relationship is positive.")

    elif r < 0:
        print("The relationship is negative.")

    else:
        print("There is no linear correlation.")

    print(f"Correlation coefficient: {r:.4f}")


    # --------------------------------------------------------
    # 12. FINAL SUMMARY
    # --------------------------------------------------------

    print("\n")
    print("============================================================")
    print("HYPOTHESIS TEST 9 — FINAL SUMMARY")
    print("============================================================")

    print("Variable 1 : Bharat_QR")
    print("Variable 2 : NPCI_Volume_Mn")
    print("Observations :", len(df_quarterly))
    print(f"Pearson r : {r:.4f}")
    print(f"P-value : {p_value:.6f}")
    print(f"Alpha : {alpha}")
    print(f"Result : {result}")
    print(f"Decision : {decision}")

    print("============================================================")

Dataset loaded successfully
Dataset shape: (12, 40)

Required columns found:
['Quarter', 'Bharat_QR', 'NPCI_Volume_Mn']

BHARAT QR CHECK

Unique values:
[6443092 6548149 6718039 6663778 6642075 6699048 6641351 6573082 6070528
 6047707 5952557 5890220]

Number of unique values:
12

Missing values:
Bharat_QR         0
NPCI_Volume_Mn    0
dtype: int64

QUARTERLY DATA FOR HYPOTHESIS TEST 9
Quarter  Bharat_QR  NPCI_Volume_Mn
     Q1   19709280   132106.807145
     Q2   20004901   141651.024411
     Q3   19284961   148237.297964
     Q4   17890484   151451.826569

Number of quarterly observations: 4

QUARTERLY VARIABLE CHECK
Bharat_QR unique quarterly values: 4
NPCI_Volume_Mn unique quarterly values: 4

Missing values:
Bharat_QR         0
NPCI_Volume_Mn    0
dtype: int64

DESCRIPTIVE STATISTICS
          Bharat_QR  NPCI_Volume_Mn
count  4.000000e+00        4.000000
mean   1.922241e+07   143361.739022
std    9.358194e+05     8540.504036
min    1.789048e+07   132106.807145
25%    1.893634e+07 

In [19]:
# ============================================================
# HYPOTHESIS TEST 10
# PSI Value Previous Month vs NPCI Value
# Pearson Correlation
# ============================================================

import pandas as pd
from scipy.stats import pearsonr

# ------------------------------------------------------------
# 1. LOAD DATASET
# ------------------------------------------------------------

file_path = r"C:\Users\THARUNI REDDY V\Downloads\analytics\ml_prepared_2025.csv"

df = pd.read_csv(file_path)

print("Dataset loaded successfully")
print("Dataset shape:", df.shape)


# ------------------------------------------------------------
# 2. CHECK REQUIRED COLUMNS
# ------------------------------------------------------------

required_columns = [
    "Quarter",
    "PSI_Value_Previous_Month",
    "NPCI_Value_Bn"
]

missing_columns = [
    col for col in required_columns
    if col not in df.columns
]

if missing_columns:
    raise ValueError(
        f"Missing required columns: {missing_columns}"
    )

print("\nRequired columns found:")
print(required_columns)


# ------------------------------------------------------------
# 3. MONTHLY VARIABLE CHECK
# ------------------------------------------------------------

print("\n============================================================")
print("PSI VALUE PREVIOUS MONTH CHECK")
print("============================================================")

print("\nUnique values:")
print(df["PSI_Value_Previous_Month"].unique())

print("\nNumber of unique values:")
print(df["PSI_Value_Previous_Month"].nunique())

print("\nMissing values:")
print(
    df[
        ["PSI_Value_Previous_Month", "NPCI_Value_Bn"]
    ].isnull().sum()
)


# ------------------------------------------------------------
# 4. CREATE QUARTERLY DATA
# ------------------------------------------------------------

df_quarterly = (
    df.groupby("Quarter", as_index=False)
      .agg({
          "PSI_Value_Previous_Month": "mean",
          "NPCI_Value_Bn": "first"
      })
)

# Correct quarter order
quarter_order = ["Q1", "Q2", "Q3", "Q4"]

df_quarterly["Quarter"] = pd.Categorical(
    df_quarterly["Quarter"],
    categories=quarter_order,
    ordered=True
)

df_quarterly = (
    df_quarterly
    .sort_values("Quarter")
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 5. DISPLAY QUARTERLY DATA
# ------------------------------------------------------------

print("\n============================================================")
print("QUARTERLY DATA FOR HYPOTHESIS TEST 10")
print("============================================================")

print(
    df_quarterly[
        [
            "Quarter",
            "PSI_Value_Previous_Month",
            "NPCI_Value_Bn"
        ]
    ].to_string(index=False)
)

print(
    "\nNumber of quarterly observations:",
    len(df_quarterly)
)


# ------------------------------------------------------------
# 6. VARIABLE CHECK
# ------------------------------------------------------------

print("\n============================================================")
print("QUARTERLY VARIABLE CHECK")
print("============================================================")

print(
    "PSI_Value_Previous_Month unique quarterly values:",
    df_quarterly["PSI_Value_Previous_Month"].nunique()
)

print(
    "NPCI_Value_Bn unique quarterly values:",
    df_quarterly["NPCI_Value_Bn"].nunique()
)


# ------------------------------------------------------------
# 7. MISSING VALUE CHECK
# ------------------------------------------------------------

print("\nMissing values:")

print(
    df_quarterly[
        [
            "PSI_Value_Previous_Month",
            "NPCI_Value_Bn"
        ]
    ].isnull().sum()
)


# ------------------------------------------------------------
# 8. DESCRIPTIVE STATISTICS
# ------------------------------------------------------------

print("\n============================================================")
print("DESCRIPTIVE STATISTICS")
print("============================================================")

print(
    df_quarterly[
        [
            "PSI_Value_Previous_Month",
            "NPCI_Value_Bn"
        ]
    ].describe()
)


# ------------------------------------------------------------
# 9. PEARSON CORRELATION
# ------------------------------------------------------------

x = df_quarterly["PSI_Value_Previous_Month"]
y = df_quarterly["NPCI_Value_Bn"]

if x.nunique() < 2 or y.nunique() < 2:

    print("\nPearson correlation cannot be calculated.")
    print("One of the variables is constant.")

else:

    r, p_value = pearsonr(x, y)

    alpha = 0.05

    print("\n============================================================")
    print("HYPOTHESIS TEST 10 — PEARSON CORRELATION")
    print("============================================================")

    print("\nResearch Question:")
    print(
        "Is there a statistically significant linear relationship "
        "between PSI Value Previous Month and NPCI Value?"
    )

    print("\nH0:")
    print(
        "There is no statistically significant linear relationship "
        "between PSI Value Previous Month and NPCI Value."
    )

    print("\nH1:")
    print(
        "There is a statistically significant linear relationship "
        "between PSI Value Previous Month and NPCI Value."
    )

    print("\nResults:")
    print(f"Pearson correlation (r): {r:.4f}")
    print(f"P-value: {p_value:.6f}")
    print(f"Significance level (alpha): {alpha}")


    # --------------------------------------------------------
    # 10. DECISION
    # --------------------------------------------------------

    if p_value < alpha:

        decision = "Reject H0"
        result = "Statistically significant"

        conclusion = (
            "There is sufficient statistical evidence of "
            "a linear relationship between PSI Value Previous "
            "Month and NPCI Value."
        )

    else:

        decision = "Fail to reject H0"
        result = "Not statistically significant"

        conclusion = (
            "There is insufficient statistical evidence of "
            "a linear relationship between PSI Value Previous "
            "Month and NPCI Value."
        )

    print("\nDecision:")
    print(decision)

    print(conclusion)


    # --------------------------------------------------------
    # 11. CORRELATION INTERPRETATION
    # --------------------------------------------------------

    print("\nCorrelation Interpretation:")

    if r > 0:
        print("The relationship is positive.")

    elif r < 0:
        print("The relationship is negative.")

    else:
        print("There is no linear correlation.")

    print(f"Correlation coefficient: {r:.4f}")


    # --------------------------------------------------------
    # 12. FINAL SUMMARY
    # --------------------------------------------------------

    print("\n")
    print("============================================================")
    print("HYPOTHESIS TEST 10 — FINAL SUMMARY")
    print("============================================================")

    print("Variable 1 : PSI_Value_Previous_Month")
    print("Variable 2 : NPCI_Value_Bn")
    print("Observations :", len(df_quarterly))
    print(f"Pearson r : {r:.4f}")
    print(f"P-value : {p_value:.6f}")
    print(f"Alpha : {alpha}")
    print(f"Result : {result}")
    print(f"Decision : {decision}")

    print("============================================================")

Dataset loaded successfully
Dataset shape: (12, 40)

Required columns found:
['Quarter', 'PSI_Value_Previous_Month', 'NPCI_Value_Bn']

PSI VALUE PREVIOUS MONTH CHECK

Unique values:
[1.87550679e+08 1.88266339e+08 1.65806011e+08 2.03299995e+08
 1.83690272e+08 1.85486574e+08 1.95375689e+08 2.01927839e+08
 1.78636985e+08 2.00643983e+08 1.98507180e+08 1.82553544e+08]

Number of unique values:
12

Missing values:
PSI_Value_Previous_Month    0
NPCI_Value_Bn               0
dtype: int64

QUARTERLY DATA FOR HYPOTHESIS TEST 10
Quarter  PSI_Value_Previous_Month  NPCI_Value_Bn
     Q1              1.805410e+08  224110.140640
     Q2              1.908256e+08  226765.475487
     Q3              1.919802e+08  245373.027942
     Q4              1.939016e+08  255516.058420

Number of quarterly observations: 4

QUARTERLY VARIABLE CHECK
PSI_Value_Previous_Month unique quarterly values: 4
NPCI_Value_Bn unique quarterly values: 4

Missing values:
PSI_Value_Previous_Month    0
NPCI_Value_Bn               

In [20]:
# ============================================================
# HYPOTHESIS TEST 11
# DEBIT CARDS vs NPCI VOLUME
# ============================================================

import pandas as pd
from scipy.stats import pearsonr

# ============================================================
# 1. LOAD VERIFIED DATASET
# ============================================================

file_path = r"C:\Users\THARUNI REDDY V\Downloads\analytics\ml_prepared_2025.csv"

df = pd.read_csv(file_path)

print("Dataset loaded successfully")
print("Dataset shape:", df.shape)

# ============================================================
# 2. REQUIRED COLUMNS
# ============================================================

required_columns = [
    "Quarter",
    "Debit_Cards",
    "NPCI_Volume_Mn"
]

missing_columns = [
    col for col in required_columns
    if col not in df.columns
]

if missing_columns:
    raise ValueError(
        f"Missing required columns: {missing_columns}"
    )

print("\nRequired columns found:")
print(required_columns)

# ============================================================
# 3. DATA CHECK
# ============================================================

print("\n" + "=" * 60)
print("DEBIT CARDS CHECK")
print("=" * 60)

print("\nUnique values:")
print(df["Debit_Cards"].unique())

print("\nNumber of unique monthly values:")
print(df["Debit_Cards"].nunique())

print("\nMissing values:")
print(
    df[
        ["Debit_Cards", "NPCI_Volume_Mn"]
    ].isnull().sum()
)

# ============================================================
# 4. CREATE QUARTERLY DATA
# ============================================================

df_quarterly = (
    df.groupby("Quarter", sort=False)
      .agg({
          "Debit_Cards": "sum",
          "NPCI_Volume_Mn": "first"
      })
      .reset_index()
)

print("\n" + "=" * 60)
print("QUARTERLY DATA FOR HYPOTHESIS TEST 11")
print("=" * 60)

print(
    df_quarterly[
        ["Quarter", "Debit_Cards", "NPCI_Volume_Mn"]
    ].to_string(index=False)
)

print(
    "\nNumber of quarterly observations:",
    len(df_quarterly)
)

# ============================================================
# 5. CHECK UNIQUE QUARTERLY VALUES
# ============================================================

print("\n" + "=" * 60)
print("QUARTERLY VARIABLE CHECK")
print("=" * 60)

print(
    "Debit_Cards unique quarterly values:",
    df_quarterly["Debit_Cards"].nunique()
)

print(
    "NPCI_Volume_Mn unique quarterly values:",
    df_quarterly["NPCI_Volume_Mn"].nunique()
)

# ============================================================
# 6. MISSING VALUE CHECK
# ============================================================

print("\nMissing values:")
print(
    df_quarterly[
        ["Debit_Cards", "NPCI_Volume_Mn"]
    ].isnull().sum()
)

# ============================================================
# 7. DESCRIPTIVE STATISTICS
# ============================================================

print("\n" + "=" * 60)
print("DESCRIPTIVE STATISTICS")
print("=" * 60)

print(
    df_quarterly[
        ["Debit_Cards", "NPCI_Volume_Mn"]
    ].describe()
)

# ============================================================
# 8. DEFINE VARIABLES
# ============================================================

x = df_quarterly["Debit_Cards"]
y = df_quarterly["NPCI_Volume_Mn"]

# ============================================================
# 9. PEARSON CORRELATION
# ============================================================

r, p_value = pearsonr(x, y)

alpha = 0.05

# ============================================================
# 10. HYPOTHESIS TEST RESULT
# ============================================================

print("\n" + "=" * 60)
print("HYPOTHESIS TEST 11 — PEARSON CORRELATION")
print("=" * 60)

print("""
Research Question:
Is there a statistically significant linear relationship
between Debit Cards and NPCI Volume?

H0:
There is no statistically significant linear relationship
between Debit Cards and NPCI Volume.

H1:
There is a statistically significant linear relationship
between Debit Cards and NPCI Volume.
""")

print("Results:")
print(f"Pearson correlation (r): {r:.4f}")
print(f"P-value: {p_value:.6f}")
print(f"Significance level (alpha): {alpha}")

# ============================================================
# 11. DECISION
# ============================================================

if p_value < alpha:
    decision = "Reject H0"
    result = (
        "There is sufficient statistical evidence of "
        "a linear relationship between the two variables."
    )
else:
    decision = "Fail to reject H0"
    result = (
        "There is insufficient statistical evidence of "
        "a linear relationship between the two variables."
    )

print("\nDecision:")
print(decision)
print(result)

# ============================================================
# 12. CORRELATION INTERPRETATION
# ============================================================

print("\nCorrelation Interpretation:")

if r > 0:
    print("The relationship is positive.")
elif r < 0:
    print("The relationship is negative.")
else:
    print("There is no linear correlation.")

print(f"Correlation coefficient: {r:.4f}")

# ============================================================
# 13. FINAL SUMMARY
# ============================================================

print("\n" + "=" * 60)
print("HYPOTHESIS TEST 11 — FINAL SUMMARY")
print("=" * 60)

print("Variable 1 : Debit_Cards")
print("Variable 2 : NPCI_Volume_Mn")
print("Observations :", len(df_quarterly))
print(f"Pearson r : {r:.4f}")
print(f"P-value : {p_value:.6f}")
print(f"Alpha : {alpha}")

if p_value < alpha:
    print("Result : Statistically significant")
else:
    print("Result : Not statistically significant")

print(f"Decision : {decision}")

print("=" * 60)

Dataset loaded successfully
Dataset shape: (12, 40)

Required columns found:
['Quarter', 'Debit_Cards', 'NPCI_Volume_Mn']

DEBIT CARDS CHECK

Unique values:
[ 982039277  985676745  990812421  995983731 1000369817 1005180283
 1012727337 1018042450 1024808790 1027150225 1031703169 1034345867]

Number of unique monthly values:
12

Missing values:
Debit_Cards       0
NPCI_Volume_Mn    0
dtype: int64

QUARTERLY DATA FOR HYPOTHESIS TEST 11
Quarter  Debit_Cards  NPCI_Volume_Mn
     Q1   2958528443   132106.807145
     Q2   3001533831   141651.024411
     Q3   3055578577   148237.297964
     Q4   3093199261   151451.826569

Number of quarterly observations: 4

QUARTERLY VARIABLE CHECK
Debit_Cards unique quarterly values: 4
NPCI_Volume_Mn unique quarterly values: 4

Missing values:
Debit_Cards       0
NPCI_Volume_Mn    0
dtype: int64

DESCRIPTIVE STATISTICS
        Debit_Cards  NPCI_Volume_Mn
count  4.000000e+00        4.000000
mean   3.027210e+09   143361.739022
std    5.926151e+07     8540.50

In [21]:
# ============================================================
# HYPOTHESIS TEST 12
# PSI VOLUME PREVIOUS MONTH vs NPCI VOLUME
# ============================================================

import pandas as pd
from scipy.stats import pearsonr

# ============================================================
# 1. LOAD VERIFIED DATASET
# ============================================================

file_path = r"C:\Users\THARUNI REDDY V\Downloads\analytics\ml_prepared_2025.csv"

df = pd.read_csv(file_path)

print("Dataset loaded successfully")
print("Dataset shape:", df.shape)

# ============================================================
# 2. REQUIRED COLUMNS
# ============================================================

required_columns = [
    "Quarter",
    "PSI_Volume_Previous_Month",
    "NPCI_Volume_Mn"
]

missing_columns = [
    col for col in required_columns
    if col not in df.columns
]

if missing_columns:
    raise ValueError(
        f"Missing required columns: {missing_columns}"
    )

print("\nRequired columns found:")
print(required_columns)

# ============================================================
# 3. VARIABLE CHECK
# ============================================================

print("\n" + "=" * 60)
print("PSI VOLUME PREVIOUS MONTH CHECK")
print("=" * 60)

print("\nUnique monthly values:")
print(df["PSI_Volume_Previous_Month"].unique())

print("\nNumber of unique monthly values:")
print(df["PSI_Volume_Previous_Month"].nunique())

print("\nMissing values:")
print(
    df[
        ["PSI_Volume_Previous_Month", "NPCI_Volume_Mn"]
    ].isnull().sum()
)

# ============================================================
# 4. CREATE QUARTERLY DATA
# ============================================================

df_quarterly = (
    df.groupby("Quarter", sort=False)
      .agg({
          "PSI_Volume_Previous_Month": "mean",
          "NPCI_Volume_Mn": "first"
      })
      .reset_index()
)

print("\n" + "=" * 60)
print("QUARTERLY DATA FOR HYPOTHESIS TEST 12")
print("=" * 60)

print(
    df_quarterly[
        [
            "Quarter",
            "PSI_Volume_Previous_Month",
            "NPCI_Volume_Mn"
        ]
    ].to_string(index=False)
)

print(
    "\nNumber of quarterly observations:",
    len(df_quarterly)
)

# ============================================================
# 5. QUARTERLY VARIABLE CHECK
# ============================================================

print("\n" + "=" * 60)
print("QUARTERLY VARIABLE CHECK")
print("=" * 60)

print(
    "PSI_Volume_Previous_Month unique quarterly values:",
    df_quarterly["PSI_Volume_Previous_Month"].nunique()
)

print(
    "NPCI_Volume_Mn unique quarterly values:",
    df_quarterly["NPCI_Volume_Mn"].nunique()
)

# ============================================================
# 6. MISSING VALUE CHECK
# ============================================================

print("\nMissing values:")
print(
    df_quarterly[
        [
            "PSI_Volume_Previous_Month",
            "NPCI_Volume_Mn"
        ]
    ].isnull().sum()
)

# ============================================================
# 7. DESCRIPTIVE STATISTICS
# ============================================================

print("\n" + "=" * 60)
print("DESCRIPTIVE STATISTICS")
print("=" * 60)

print(
    df_quarterly[
        [
            "PSI_Volume_Previous_Month",
            "NPCI_Volume_Mn"
        ]
    ].describe()
)

# ============================================================
# 8. DEFINE VARIABLES
# ============================================================

x = df_quarterly["PSI_Volume_Previous_Month"]
y = df_quarterly["NPCI_Volume_Mn"]

# ============================================================
# 9. PEARSON CORRELATION
# ============================================================

r, p_value = pearsonr(x, y)

alpha = 0.05

# ============================================================
# 10. HYPOTHESIS TEST
# ============================================================

print("\n" + "=" * 60)
print("HYPOTHESIS TEST 12 — PEARSON CORRELATION")
print("=" * 60)

print("""
Research Question:
Is there a statistically significant linear relationship
between PSI Volume Previous Month and NPCI Volume?

H0:
There is no statistically significant linear relationship
between PSI Volume Previous Month and NPCI Volume.

H1:
There is a statistically significant linear relationship
between PSI Volume Previous Month and NPCI Volume.
""")

print("Results:")
print(f"Pearson correlation (r): {r:.4f}")
print(f"P-value: {p_value:.6f}")
print(f"Significance level (alpha): {alpha}")

# ============================================================
# 11. DECISION
# ============================================================

if p_value < alpha:
    decision = "Reject H0"
    result = (
        "There is sufficient statistical evidence of "
        "a linear relationship between the two variables."
    )
else:
    decision = "Fail to reject H0"
    result = (
        "There is insufficient statistical evidence of "
        "a linear relationship between the two variables."
    )

print("\nDecision:")
print(decision)
print(result)

# ============================================================
# 12. CORRELATION INTERPRETATION
# ============================================================

print("\nCorrelation Interpretation:")

if r > 0:
    print("The relationship is positive.")
elif r < 0:
    print("The relationship is negative.")
else:
    print("There is no linear correlation.")

print(f"Correlation coefficient: {r:.4f}")

# ============================================================
# 13. FINAL SUMMARY
# ============================================================

print("\n" + "=" * 60)
print("HYPOTHESIS TEST 12 — FINAL SUMMARY")
print("=" * 60)

print("Variable 1 : PSI_Volume_Previous_Month")
print("Variable 2 : NPCI_Volume_Mn")
print("Observations :", len(df_quarterly))
print(f"Pearson r : {r:.4f}")
print(f"P-value : {p_value:.6f}")
print(f"Alpha : {alpha}")

if p_value < alpha:
    print("Result : Statistically significant")
else:
    print("Result : Not statistically significant")

print(f"Decision : {decision}")

print("=" * 60)

Dataset loaded successfully
Dataset shape: (12, 40)

Required columns found:
['Quarter', 'PSI_Volume_Previous_Month', 'NPCI_Volume_Mn']

PSI VOLUME PREVIOUS MONTH CHECK

Unique monthly values:
[ 998634.12449    1012778.24986059  962974.0919     1093244.38466
 1056516.12144824 1102932.03207    1084787.20758    1147549.08217886
 1184449.72459    1158441.06408    1218450.10893772 1203626.45730343]

Number of unique monthly values:
12

Missing values:
PSI_Volume_Previous_Month    0
NPCI_Volume_Mn               0
dtype: int64

QUARTERLY DATA FOR HYPOTHESIS TEST 12
Quarter  PSI_Volume_Previous_Month  NPCI_Volume_Mn
     Q1               9.914622e+05   132106.807145
     Q2               1.084231e+06   141651.024411
     Q3               1.138929e+06   148237.297964
     Q4               1.193506e+06   151451.826569

Number of quarterly observations: 4

QUARTERLY VARIABLE CHECK
PSI_Volume_Previous_Month unique quarterly values: 4
NPCI_Volume_Mn unique quarterly values: 4

Missing values:
PSI_

In [22]:
# ============================================================
# HYPOTHESIS TEST 13
# PSI VOLUME PREVIOUS MONTH vs NPCI VALUE
# ============================================================

import pandas as pd
from scipy.stats import pearsonr

# ============================================================
# 1. LOAD VERIFIED DATASET
# ============================================================

file_path = r"C:\Users\THARUNI REDDY V\Downloads\analytics\ml_prepared_2025.csv"

df = pd.read_csv(file_path)

print("Dataset loaded successfully")
print("Dataset shape:", df.shape)

# ============================================================
# 2. REQUIRED COLUMNS
# ============================================================

required_columns = [
    "Quarter",
    "PSI_Volume_Previous_Month",
    "NPCI_Value_Bn"
]

missing_columns = [
    col for col in required_columns
    if col not in df.columns
]

if missing_columns:
    raise ValueError(
        f"Missing required columns: {missing_columns}"
    )

print("\nRequired columns found:")
print(required_columns)

# ============================================================
# 3. VARIABLE CHECK
# ============================================================

print("\n" + "=" * 60)
print("PSI VOLUME PREVIOUS MONTH CHECK")
print("=" * 60)

print("\nUnique monthly values:")
print(df["PSI_Volume_Previous_Month"].unique())

print("\nNumber of unique monthly values:")
print(df["PSI_Volume_Previous_Month"].nunique())

print("\nMissing values:")
print(
    df[
        ["PSI_Volume_Previous_Month", "NPCI_Value_Bn"]
    ].isnull().sum()
)

# ============================================================
# 4. CREATE QUARTERLY DATA
# ============================================================

df_quarterly = (
    df.groupby("Quarter", sort=False)
      .agg({
          "PSI_Volume_Previous_Month": "mean",
          "NPCI_Value_Bn": "first"
      })
      .reset_index()
)

print("\n" + "=" * 60)
print("QUARTERLY DATA FOR HYPOTHESIS TEST 13")
print("=" * 60)

print(
    df_quarterly[
        [
            "Quarter",
            "PSI_Volume_Previous_Month",
            "NPCI_Value_Bn"
        ]
    ].to_string(index=False)
)

print(
    "\nNumber of quarterly observations:",
    len(df_quarterly)
)

# ============================================================
# 5. QUARTERLY VARIABLE CHECK
# ============================================================

print("\n" + "=" * 60)
print("QUARTERLY VARIABLE CHECK")
print("=" * 60)

print(
    "PSI_Volume_Previous_Month unique quarterly values:",
    df_quarterly["PSI_Volume_Previous_Month"].nunique()
)

print(
    "NPCI_Value_Bn unique quarterly values:",
    df_quarterly["NPCI_Value_Bn"].nunique()
)

# ============================================================
# 6. MISSING VALUE CHECK
# ============================================================

print("\nMissing values:")
print(
    df_quarterly[
        [
            "PSI_Volume_Previous_Month",
            "NPCI_Value_Bn"
        ]
    ].isnull().sum()
)

# ============================================================
# 7. DESCRIPTIVE STATISTICS
# ============================================================

print("\n" + "=" * 60)
print("DESCRIPTIVE STATISTICS")
print("=" * 60)

print(
    df_quarterly[
        [
            "PSI_Volume_Previous_Month",
            "NPCI_Value_Bn"
        ]
    ].describe()
)

# ============================================================
# 8. DEFINE VARIABLES
# ============================================================

x = df_quarterly["PSI_Volume_Previous_Month"]
y = df_quarterly["NPCI_Value_Bn"]

# ============================================================
# 9. PEARSON CORRELATION
# ============================================================

r, p_value = pearsonr(x, y)

alpha = 0.05

# ============================================================
# 10. HYPOTHESIS TEST
# ============================================================

print("\n" + "=" * 60)
print("HYPOTHESIS TEST 13 — PEARSON CORRELATION")
print("=" * 60)

print("""
Research Question:
Is there a statistically significant linear relationship
between PSI Volume Previous Month and NPCI Value?

H0:
There is no statistically significant linear relationship
between PSI Volume Previous Month and NPCI Value.

H1:
There is a statistically significant linear relationship
between PSI Volume Previous Month and NPCI Value.
""")

print("Results:")
print(f"Pearson correlation (r): {r:.4f}")
print(f"P-value: {p_value:.6f}")
print(f"Significance level (alpha): {alpha}")

# ============================================================
# 11. DECISION
# ============================================================

if p_value < alpha:

    decision = "Reject H0"

    result = (
        "There is sufficient statistical evidence of "
        "a linear relationship between the two variables."
    )

else:

    decision = "Fail to reject H0"

    result = (
        "There is insufficient statistical evidence of "
        "a linear relationship between the two variables."
    )

print("\nDecision:")
print(decision)
print(result)

# ============================================================
# 12. CORRELATION INTERPRETATION
# ============================================================

print("\nCorrelation Interpretation:")

if r > 0:
    print("The relationship is positive.")

elif r < 0:
    print("The relationship is negative.")

else:
    print("There is no linear correlation.")

print(f"Correlation coefficient: {r:.4f}")

# ============================================================
# 13. FINAL SUMMARY
# ============================================================

print("\n" + "=" * 60)
print("HYPOTHESIS TEST 13 — FINAL SUMMARY")
print("=" * 60)

print("Variable 1 : PSI_Volume_Previous_Month")
print("Variable 2 : NPCI_Value_Bn")
print("Observations :", len(df_quarterly))
print(f"Pearson r : {r:.4f}")
print(f"P-value : {p_value:.6f}")
print(f"Alpha : {alpha}")

if p_value < alpha:
    print("Result : Statistically significant")
else:
    print("Result : Not statistically significant")

print(f"Decision : {decision}")

print("=" * 60)

Dataset loaded successfully
Dataset shape: (12, 40)

Required columns found:
['Quarter', 'PSI_Volume_Previous_Month', 'NPCI_Value_Bn']

PSI VOLUME PREVIOUS MONTH CHECK

Unique monthly values:
[ 998634.12449    1012778.24986059  962974.0919     1093244.38466
 1056516.12144824 1102932.03207    1084787.20758    1147549.08217886
 1184449.72459    1158441.06408    1218450.10893772 1203626.45730343]

Number of unique monthly values:
12

Missing values:
PSI_Volume_Previous_Month    0
NPCI_Value_Bn                0
dtype: int64

QUARTERLY DATA FOR HYPOTHESIS TEST 13
Quarter  PSI_Volume_Previous_Month  NPCI_Value_Bn
     Q1               9.914622e+05  224110.140640
     Q2               1.084231e+06  226765.475487
     Q3               1.138929e+06  245373.027942
     Q4               1.193506e+06  255516.058420

Number of quarterly observations: 4

QUARTERLY VARIABLE CHECK
PSI_Volume_Previous_Month unique quarterly values: 4
NPCI_Value_Bn unique quarterly values: 4

Missing values:
PSI_Volume_

In [23]:
import pandas as pd

# ============================================================
# HYPOTHESIS TESTING — FINAL SUMMARY (TESTS 1–13)
# ============================================================

results = [
    [1, "DC ATM Withdrawal Volume", "NPCI Volume", -0.9069, 0.093085],
    [2, "DC ATM Withdrawal Volume", "NPCI Value", -0.6615, 0.338524],
    [3, "POS", "NPCI Volume", 0.9006, 0.099425],
    [4, "POS", "NPCI Value", 0.6943, 0.305690],
    [5, "Micro ATM", "NPCI Volume", -0.7709, 0.229130],
    [6, "Micro ATM", "NPCI Value", -0.8764, 0.123637],
    [7, "DC Online Volume", "NPCI Volume", -0.9148, 0.085160],
    [8, "UPI QR", "NPCI Volume", 0.9627, 0.037266],
    [9, "Bharat QR", "NPCI Volume", -0.7211, 0.278948],
    [10, "PSI Value Previous Month", "NPCI Value", 0.7580, 0.242031],
    [11, "Debit Cards", "NPCI Volume", 0.9807, 0.019275],
    [12, "PSI Volume Previous Month", "NPCI Volume", 0.9942, 0.005778],
    [13, "PSI Volume Previous Month", "NPCI Value", 0.9275, 0.072545]
]

df_hypothesis = pd.DataFrame(
    results,
    columns=[
        "Test_No",
        "Variable_1",
        "Variable_2",
        "Pearson_r",
        "P_value"
    ]
)

alpha = 0.05

# Decision
df_hypothesis["Alpha"] = alpha
df_hypothesis["Result"] = df_hypothesis["P_value"].apply(
    lambda p: "Statistically Significant" if p < alpha
    else "Not Statistically Significant"
)

df_hypothesis["Decision"] = df_hypothesis["P_value"].apply(
    lambda p: "Reject H0" if p < alpha
    else "Fail to Reject H0"
)

# Direction
df_hypothesis["Relationship"] = df_hypothesis["Pearson_r"].apply(
    lambda r: "Positive" if r > 0
    else "Negative"
)

# Strength
def correlation_strength(r):
    r_abs = abs(r)

    if r_abs >= 0.90:
        return "Very Strong"
    elif r_abs >= 0.70:
        return "Strong"
    elif r_abs >= 0.50:
        return "Moderate"
    elif r_abs >= 0.30:
        return "Weak"
    else:
        return "Very Weak"

df_hypothesis["Correlation_Strength"] = (
    df_hypothesis["Pearson_r"].apply(correlation_strength)
)

# Display
print("=" * 100)
print("FINAL HYPOTHESIS TESTING SUMMARY — TESTS 1–13")
print("=" * 100)

print(df_hypothesis.to_string(index=False))

print("\n" + "=" * 100)
print("SIGNIFICANT RESULTS")
print("=" * 100)

significant = df_hypothesis[
    df_hypothesis["P_value"] < alpha
]

print(significant.to_string(index=False))

print("\n" + "=" * 100)
print("SUMMARY")
print("=" * 100)

print("Total hypothesis tests:", len(df_hypothesis))
print("Statistically significant:", len(significant))
print(
    "Not statistically significant:",
    len(df_hypothesis) - len(significant)
)

# Save
output_file = (
    r"C:\Users\THARUNI REDDY V\Downloads\analytics"
    r"\hypothesis_testing_summary_tests_1_13_2025.csv"
)

df_hypothesis.to_csv(output_file, index=False)

print("\nSaved:")
print(output_file)
print("=" * 100)

FINAL HYPOTHESIS TESTING SUMMARY — TESTS 1–13
 Test_No                Variable_1  Variable_2  Pearson_r  P_value  Alpha                        Result          Decision Relationship Correlation_Strength
       1  DC ATM Withdrawal Volume NPCI Volume    -0.9069 0.093085   0.05 Not Statistically Significant Fail to Reject H0     Negative          Very Strong
       2  DC ATM Withdrawal Volume  NPCI Value    -0.6615 0.338524   0.05 Not Statistically Significant Fail to Reject H0     Negative             Moderate
       3                       POS NPCI Volume     0.9006 0.099425   0.05 Not Statistically Significant Fail to Reject H0     Positive          Very Strong
       4                       POS  NPCI Value     0.6943 0.305690   0.05 Not Statistically Significant Fail to Reject H0     Positive             Moderate
       5                 Micro ATM NPCI Volume    -0.7709 0.229130   0.05 Not Statistically Significant Fail to Reject H0     Negative               Strong
       6          